<br>

<br>

# Autoscout EDA

## EDA for Car Price Prediction

<br>

<br>! Hello everyone ! <br><br>
This dataset contains many features of 9 different car models.
In this project we are going to prepare this dataset to provide an appropriate input to the ML models.<br>
Thus the steps we are going to follow ;
- Fix the corrupted data formats
- Handle with outliers and missing values.
- Drop the columns/rows you determined unnecessary as a result of your analysis.

<br>
<br> Before start analyzing first let's import the libraries we might use for further steps.

In [3]:
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

from collections import Counter
import warnings
warnings.filterwarnings('ignore')

<br>

In [11]:
df = pd.read_json("scout_car.json", lines = True)

Even though we ussually work on a csv file, this once we are going to be working on a json file. The way that we import it is almost the same with csv file. Only the difference is "lines" parameter.

<br>

- To get an idea of what the data looks like, let's have a look at that by getting some random samples.<br><br>

- I am going to get the tranpose of it to see all the columns.

In [12]:
df.sample(3).T

,2821,7893,1265
url,https://www.autoscout24.com//offers/audi-a3-sp...,https://www.autoscout24.com//offers/opel-astra...,https://www.autoscout24.com//offers/audi-a1-sp...
make_model,Audi A3,Opel Astra,Audi A1
short_description,SPB 1.6 TDI 110 CV Stronic--06/22445431--,ST 1.4 Turbo Innovation Navi/Lane-Assi/4xSHZ/Kame,Sportback 1.0 TFSI *Sitzheizung*Einparkhilfe*
body_type,Sedans,Station wagon,Sedans
price,16999,19480,15495
vat,VAT deductible,VAT deductible,VAT deductible
km,"60,000 km","13,200 km","11,500 km"
registration,01/2017,01/2019,03/2018
prev_owner,None,1 previous owner,1 previous owner
kW,NaN,NaN,NaN


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15919 entries, 0 to 15918
Data columns (total 54 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   url                            15919 non-null  object 
 1   make_model                     15919 non-null  object 
 2   short_description              15873 non-null  object 
 3   body_type                      15859 non-null  object 
 4   price                          15919 non-null  int64  
 5   vat                            11406 non-null  object 
 6   km                             15919 non-null  object 
 7   registration                   15919 non-null  object 
 8   prev_owner                     9091 non-null   object 
 9   kW                             0 non-null      float64
 10  hp                             15919 non-null  object 
 11  Type                           15917 non-null  object 
 12  Previous Owners                9279 non-null  

In [44]:
df.notnull().sum()

url                              15919
make_model                       15919
short_description                15873
body_type                        15859
price                            15919
vat                              11406
km                               15919
registration                     15919
prev_owner                        9091
kW                                   0
hp                               15919
Type                             15917
Previous Owners                   9279
Next Inspection                   3535
Inspection new                    3932
Warranty                         10499
Full Service                      8215
Non-smoking Vehicle               7177
null                             15919
Make                             15919
Model                            15919
Offer Number                     12744
First Registration               14322
Body Color                       15322
Paint Type                       10147
Body Color Original      

<br>I am not happy to say that but this dataset is __chaotic__ !! We have got a lot to do, better to get started.<br><br>

As always before start to cleaning I am going to make a "To Do List" by shooting a glance to each column one by one.

In [16]:
df1 = df.copy()

In [39]:
df1.Warranty.value_counts().head(10)

TypeError: unhashable type: 'list'

Exception ignored in: 'pandas._libs.index.IndexEngine._call_map_locations'
Traceback (most recent call last):
  File "pandas\_libs\hashtable_class_helper.pxi", line 4588, in pandas._libs.hashtable.PyObjectHashTable.map_locations
TypeError: unhashable type: 'list'


[\n, \n, \nEuro 6\n]           1868
\n12 months\n                  1177
\n                              979
\n24 months\n                   566
[\n, \n, \n4 (Green)\n]         517
[\n12 months\n, \nEuro 6\n]     425
[\n, \n]                        364
[\n, \n, \nEuro 6d-TEMP\n]      362
[\n12 months\n]                 128
\n48 months\n                   109
Name: Warranty, dtype: int64

In [46]:
df1['Body Color Original'].value_counts()


TypeError: unhashable type: 'list'

Exception ignored in: 'pandas._libs.index.IndexEngine._call_map_locations'
Traceback (most recent call last):
  File "pandas\_libs\hashtable_class_helper.pxi", line 4588, in pandas._libs.hashtable.PyObjectHashTable.map_locations
TypeError: unhashable type: 'list'


[\nOnyx Schwarz\n]              338
[\nBianco\n]                    282
[\nMythosschwarz Metallic\n]    238
[\nBrillantschwarz\n]           216
[\nGris\n]                      204
                               ... 
[\nBleu Celeste\n]                1
[\nBlu cosmo metallizzato\n]      1
[\nBlac Ivoire D16\n]             1
[\nkarbonschwarz met\n]           1
[\nSATIN STEEL\n]                 1
Name: Body Color Original, Length: 1927, dtype: int64

## To Do List

1. 'url' column has no relation to the price column. We can drop it.
2. In 'short_description' column there are so many info and each column is different from another which makes it harder to clasify them. We can drop it.
3. Change the dtype of km column to integer or float.
4. For 'registration' column remove the months and change the column name as registration year. Look at first reg column as well, create age column.
5. Change the prev_owner column's data type to integer and drop the other previous owners column.
6. Drop the 'kW' column since it doesn't have any non null value.
7. Change the 'hp' column name to the 'kW' also the dtype to integer.
8. Change 'Type' column as 'Fuel_Type' and remove the unnecessary informations.
9. For the 'Next Inspectation" column revo everything bt the year and month inf.
10. For the ins new column change the dtype to bool.
11. Drop 'Full service' column.
12. Drop non smoking, null, make, model and offer number columns.
13. Drop the bofy color original.
14. 